In [ ]:
import numpy as np
import msicpe.san as san
from plotly import express as px
import scipy.signal as signal
import pandas as pd


def vecteur (H, fichier) : #fonction qui renvoie 
    v = [] #vecteur qui décrira le signal

    signal_dict = np.load(fichier) #récupération du signal
    s = signal_dict['s'] 
    t = signal_dict['t']

    nu, S = san.trans_fourier(s, t) #calcul de la transformée de fourrier

    S_norm = np.abs(S) / np.max(np.abs(S)) #normalisation du signal

    nu_fond = san.detect_fondamentale(S_norm, nu) #récupération de la fréquence fondamentale
    i_fond = np.argmin(np.abs(nu - nu_fond)) #récupération de l'indice de la fréquence connue du signal
    amp_fond = S_norm[i_fond]                #la plus proche de la fréquence fondamentale
    v.append(amp_fond) #ajout de l'amplitude à la fréquence fondamentale à notre vecteur

    freq_dint = [] #récupération des fréquences d'intérets autre que la fondamentale
    for i in range(2, H+2) :
        freq_dint.append(i * nu_fond)

    freq_pic, amp_pic = san.detect_pics(S_norm, freq_dint, nu) #récupération des fréquences connues proches des
                                                               #fréquences d'intérêt et leurs amplitudes
    v.extend(amp_pic) #création du vecteur final
    v = np.log10(v) #conversion avec le log 10
    return v

def moyenne(list_vecteurs) : #fonction qui donne le vecteur moyen de plusieurs vecteurs d'une liste
    vect_moyen = []

    nb_vecteur = len(list_vecteurs)

    for i in range(len(list_vecteurs[0])) :
        somme = 0
        for k in range(nb_vecteur) :
            somme = somme + list_vecteurs[k][i]
        vect_moyen.append(somme / nb_vecteur)
    return vect_moyen


def vecteur_ref (H, signaux) : #fonction qui donne le vecteur de référence d'une liste de signaux
    list_vect_sign = []
    for signal in signaux :
        list_vect_sign.append(vecteur(H, signal))
    vect_ref = moyenne(list_vect_sign)
    return vect_ref

def distance(vect1, vect2): #fonction qui donne la distance entre deux vecteurs
    long_vect = len(vect1)
    somme = 0
    for i in range(long_vect) :
          somme = somme + (vect1[i] - vect2[i])**2
    return somme**0.5

#trouvons le H qu'il nous faut
signaux_apprentissage_a = ['data partie 2-20261112/apprentissage/a_1.npz', 'data partie 2-20261112/apprentissage/a_2.npz', 'data partie 2-20261112/apprentissage/a_3.npz', 'data partie 2-20261112/apprentissage/a_4.npz', 'data partie 2-20261112/apprentissage/a_5.npz']
signaux_apprentissage_e = ['data partie 2-20261112/apprentissage/e_1.npz', 'data partie 2-20261112/apprentissage/e_2.npz', 'data partie 2-20261112/apprentissage/e_3.npz', 'data partie 2-20261112/apprentissage/e_4.npz', 'data partie 2-20261112/apprentissage/e_5.npz']
signaux_apprentissage_i = ['data partie 2-20261112/apprentissage/i_1.npz', 'data partie 2-20261112/apprentissage/i_2.npz', 'data partie 2-20261112/apprentissage/i_3.npz', 'data partie 2-20261112/apprentissage/i_4.npz', 'data partie 2-20261112/apprentissage/i_5.npz']
signaux_apprentissage_o = ['data partie 2-20261112/apprentissage/o_1.npz', 'data partie 2-20261112/apprentissage/o_2.npz', 'data partie 2-20261112/apprentissage/o_3.npz', 'data partie 2-20261112/apprentissage/o_4.npz', 'data partie 2-20261112/apprentissage/o_5.npz']
signaux_apprentissage_u = ['data partie 2-20261112/apprentissage/u_1.npz', 'data partie 2-20261112/apprentissage/u_2.npz', 'data partie 2-20261112/apprentissage/u_3.npz', 'data partie 2-20261112/apprentissage/u_4.npz', 'data partie 2-20261112/apprentissage/u_5.npz']

signaux_apprentissage = [signaux_apprentissage_a, signaux_apprentissage_e, signaux_apprentissage_i, signaux_apprentissage_o, signaux_apprentissage_u]

signaux_test_a = ['data partie 2-20261112/test/a_6.npz', 'data partie 2-20261112/test/a_7.npz']
signaux_test_e = ['data partie 2-20261112/test/e_6.npz', 'data partie 2-20261112/test/e_7.npz']
signaux_test_i = ['data partie 2-20261112/test/i_6.npz', 'data partie 2-20261112/test/i_7.npz']
signaux_test_o = ['data partie 2-20261112/test/o_6.npz', 'data partie 2-20261112/test/o_7.npz']
signaux_test_u = ['data partie 2-20261112/test/u_6.npz', 'data partie 2-20261112/test/u_7.npz']

signaux_test = [signaux_test_a, signaux_test_e, signaux_test_i, signaux_test_o, signaux_test_u]

precisions = []

for i in range(1,11) : #test des H de 1 à 10
    mat_conf = [[0, 0, 0, 0, 0],  #pour cela on crée une matrice de confusion qui déterminera la précision pour chaque H
                [0, 0, 0, 0, 0],
                [0, 0, 0, 0, 0],
                [0, 0, 0, 0, 0],
                [0, 0, 0, 0, 0]]

    vect_ref = [] #liste des vecteurs de référence de chaque voyelle pour un H donné
    for vect in signaux_apprentissage : 
        vect_ref.append(vecteur_ref(i, vect))

    for k in range(len(signaux_test)) : #test de toutes les voyelles de test
        for signal in signaux_test[k] : #test de chaque signal pour une voyelle
            distances = []
            vect_test = vecteur(i, signal) #le vecteur associé au signal
            for vect in vect_ref : #on calcule la distance de ce vecteur à chaque vecteur de référence
                distances.append(distance(vect_test, vect))
            indice_lettre_supposee = np.argmin(distances) #indice de la distance la plus petite, correspondant donc à la voyelle que le programme pense correspondre au signal
            mat_conf[k][indice_lettre_supposee] += 1 #ajout du resultat dans la matrice de confusion

    vrai = 0
    faux = 0
    for k in range(len(mat_conf)) : #calcul de la précison de chaque matrice de confusion donc la précision associée à chaque H
        for n in range(len(mat_conf[0])) :
            if k == n :
                vrai = vrai + mat_conf[k][n]
            else : 
                faux = faux + mat_conf[k][n]
    
    precisions.append(vrai / (vrai+faux))

meilleur_H = np.argmax(precisions) + 1 #H correspondant à la meilleure précisions parmi les H testés

signal_etude = 'data partie 2-20261112/etude/signal_5.npz' #récupération du signal à étudier
voyelles = ["a", "e", "i", "o", "u"]
vect_ref = [] #liste des vecteurs de référence pour le meilleur H possible
for vect in signaux_apprentissage :
    vect_ref.append(vecteur_ref(meilleur_H, vect))

distances = []
vect_etude = vecteur(meilleur_H, signal_etude) #vecteur associé au signal à étudier
for vect in vect_ref : #calcul des distances de ce vecteur aux vecteurs de référence
    distances.append(distance(vect_etude, vect))
indice_lettre_supposee = np.argmin(distances) #indice de la plus petite distance, associé à la voyelle que le programme pense être celle du signal d'étude

print(voyelles[indice_lettre_supposee])


e
